# Tutorial 2: Simulated Experiments Walkthrough

This notebook reproduces all 8 simulated experiments from the paper:
1. Manifold dimensionality
2. Trajectory geometry visualization
3. Geodesic length analysis
4. Curvature profiles
5. Phase detection
6. SRVF time-warp invariance
7. Inter-structure synchrony
8. Phenotype classification

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from core.trajectory import SwallowingTrajectory
from core.metrics import extract_all_metrics
from core.srvf import srvf_distance_with_alignment, time_warp_invariance_test
from core.phase_detection import detect_phases_geometric, bottleneck_traversal_score
from core.phenotype import PhenotypeClassifier
from simulation.trajectory_generator import SyntheticManifold, TrajectoryGenerator, DEFAULT_LANDMARK_GROUPS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Generate full cohort
sm = SyntheticManifold(intrinsic_dim=5, ambient_dim=28, seed=42)
gen = TrajectoryGenerator(sm, n_frames=100, fps=25.0, seed=42)
conditions = ['healthy','fibrotic','weak','compensatory','neurogenic']
cohort = gen.generate_cohort(n_per_condition=50, conditions=conditions)
groups = DEFAULT_LANDMARK_GROUPS

print(f'Generated {len(cohort)} trajectories')

## Experiment 1: Manifold Dimensionality

In [ ]:
X = np.vstack([t.landmarks for t in cohort])
pca = PCA(n_components=15).fit(X)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 16), cumvar, 'o-', color='tomato')
ax.axhline(0.99, color='gray', ls='--')
ax.axvline(5, color='red', ls='--', label='True d=5')
ax.set_xlabel('Components'); ax.set_ylabel('Cumulative Variance')
ax.set_title('Manifold Dimensionality Estimation'); ax.legend()
plt.tight_layout(); plt.show()
print(f'd=5: {cumvar[4]:.4f}')

## Experiment 2: 3D Trajectory Visualization

In [ ]:
pca3 = PCA(n_components=3).fit(X)
colors = {'healthy':'#2ecc71','fibrotic':'#e74c3c','weak':'#3498db','compensatory':'#f39c12','neurogenic':'#9b59b6'}

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
for cond in conditions:
    for t in [t for t in cohort if t.condition==cond][:6]:
        e = pca3.transform(t.landmarks)
        ax.plot(e[:,0], e[:,1], e[:,2], color=colors[cond], alpha=0.4, lw=0.8)
    ax.plot([],[],[], color=colors[cond], lw=2.5, label=cond.capitalize())
ax.legend(); ax.view_init(25, 45)
ax.set_title('Swallowing Trajectories in PCA Space'); plt.show()

## Experiment 3–4: Geodesic Length & Curvature

In [ ]:
data = [{'condition': t.condition, 'L': t.smooth().arc_length(),
         'kappa': np.mean(t.smooth().curvature)} for t in cohort]
df = pd.DataFrame(data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x='condition', y='L', hue='condition', order=conditions, palette=colors, ax=ax1, legend=False)
ax1.set_title('Geodesic Length by Phenotype'); ax1.set_ylabel('L(γ)')
sns.boxplot(data=df, x='condition', y='kappa', hue='condition', order=conditions, palette=colors, ax=ax2, legend=False)
ax2.set_title('Mean Curvature by Phenotype'); ax2.set_ylabel('κ')
plt.tight_layout(); plt.show()

## Experiment 5: Phase Detection

In [ ]:
h = [t for t in cohort if t.condition=='healthy'][0].smooth().interpolate(200)
phases = detect_phases_geometric(h)
pc = ['#1abc9c','#e74c3c','#9b59b6','#f39c12','#3498db']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(h.time, h.speed, 'k-', lw=1.5)
for i, (name, (ts, te)) in enumerate(phases.items()):
    ax1.axvspan(ts, te, alpha=0.2, color=pc[i], label=name.replace('_',' ').title())
ax1.legend(fontsize=7, ncol=3); ax1.set_ylabel('Speed'); ax1.set_title('Phase Detection')
ax2.plot(h.time, h.curvature, 'k-', lw=1.5)
for i, (name, (ts, te)) in enumerate(phases.items()):
    ax2.axvspan(ts, te, alpha=0.2, color=pc[i])
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Curvature')
plt.tight_layout(); plt.show()
print(f'Detected {len(phases)} phases')

## Experiment 6: SRVF Invariance

In [ ]:
h_traj = [t for t in cohort if t.condition=='healthy'][0].smooth()
inv = time_warp_invariance_test(h_traj, n_warpings=10, warp_strength=0.3)
print(f'SRVF dist to warped self:      {np.mean(inv["srvf_distances"]):.4f} ± {np.std(inv["srvf_distances"]):.4f}')
print(f'Euclidean dist to warped self:  {np.mean(inv["euclidean_distances"]):.4f} ± {np.std(inv["euclidean_distances"]):.4f}')

## Experiment 7–8: Synchrony & Classification

In [ ]:
clf = PhenotypeClassifier(method='random_forest')
feat_df = clf.extract_features(cohort, groups)
Xf = np.nan_to_num(feat_df[clf.feature_names_].values)
yf = feat_df['condition'].map({c: i for i, c in enumerate(conditions)}).values

cv = clf.cross_validate(Xf, yf, n_folds=5)
print(f'5-fold CV accuracy: {cv["mean_accuracy"]:.1%} ± {cv["std_accuracy"]:.1%}')

clf.fit(Xf, yf)
imp = pd.Series(clf.feature_importances_, index=clf.feature_names_).sort_values(ascending=False)
print('\nTop 10 features:')
for name, val in imp.head(10).items():
    print(f'  {name:<35s} {val:.1%}')